倒入工具包
    使用import把库一次性倒入。其中LinearRegression和LogisticRegression是本次homework中的真实模型

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.metrics import (
    r2_score, mean_absolute_error, mean_squared_error,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix,
)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (7, 4)

读原始数据
    用仓库根目录的相对路径sections/s02/homework/openpoly_calculated_properties.csv读取原始数据

In [4]:
DATA_PATH = "sections/s02/homework/openpoly_calculated_properties.csv"
df = pd.read_csv(DATA_PATH)

print("shape:", df.shape)
print("columns:", list(df.columns))
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'sections/s02/homework/openpoly_calculated_properties.csv'

有效性规则
    != 0 逐元素比较，得到 True/False 布尔掩码。
    df["Energy_Hatree"] 只留 True 的行。
    copy() 复制，避免之后改动牵连原 df的数据。
    reset_index(drop=True) 行号重排 0,1,2…，drop=True 丢掉旧索引。
    最终跑完代码应得 removed 74、valid 926。    

In [ ]:
raw_rows = len(df)
model_df = df[df["Energy_Hatree"] != 0].copy().reset_index(drop=True)
removed_rows = raw_rows - len(model_df)
print("raw rows:", raw_rows)
print("removed (Energy_Hatree == 0):", removed_rows)
print("valid rows:", len(model_df))

定义目标与诚实特征
    排除 id（编号）、目标自己、HOMO_eV/LUMO_eV（公式泄漏：gap = LUMO - HOMO）

In [ ]:
TARGET = "HOMO_LUMO_Gap_eV"
EXCLUDED_NUMERIC = ["id", TARGET, "HOMO_eV", "LUMO_eV"]
HONEST_FEATURES = [
    "Energy_Hatree", "es", "Isotropic_Polarizability_au",
    "Inner_energy_correction_Hatree",
    "Thermal_correction_to_Enthalpy_Hatree",
    "Thermal_correction_to_Gibbs_Free_Energy_Hatree",
    "Entropy_Hatree", "Dipole_Debye",
    "Gibbs_Free_Energy_Hatree", "Enthalpy_Hatree",
]
print("target:", TARGET)
print("number of honest features:", len(HONEST_FEATURES))
print("excluded numeric columns:", EXCLUDED_NUMERIC)

80/20 划分
    用列名列表取多列 → DataFrame；取一列 → Series。
    train_test_split 把 X、y 成对切分，返回 4 个值，用解包一次赋值。
    set(A) & set(B) 取交集，长度应为 0，证明无重叠。
    跑完该段代码应得 741/185。

In [ ]:
X = model_df[HONEST_FEATURES]
y = model_df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,
)
print("train rows:", len(X_train))
print("test rows :", len(X_test))
print("index overlap:", len(set(X_train.index) & set(X_test.index)))

NameError: name 'model_df' is not defined

对目标进行分布
    describe(...) 出 count/mean/std/min/分位数/max。
    isna().sum() 数缺失。
    subplots(1, 2, ...) 建 1 行 2 列子图，返回画布和坐标轴数组。
    ax=axes[0] 决定画在哪张子图；
    bins=30 分 30 桶。
    tight_layout() 自动调间距
    show() 显示。

In [ ]:
gap = model_df[TARGET]
desc = gap.describe(percentiles=[0.25, 0.5, 0.75])
print(desc)
print("missing:", gap.isna().sum())
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(gap, bins=30, ax=axes[0])
axes[0].set_title("Target histogram: HOMO_LUMO_Gap_eV")
sns.boxplot(x=gap, ax=axes[1])
axes[1].set_title("Target box plot")
plt.tight_layout()
plt.show()

质量表
    键=列名，值=列内容
    "dtype"数据类型
    "missing"缺失数据
    "min/max"范围

In [ ]:
audit = pd.DataFrame({
    "dtype": model_df[HONEST_FEATURES].dtypes.astype(str),
    "missing": model_df[HONEST_FEATURES].isna().sum(),
    "min": model_df[HONEST_FEATURES].min(),
    "max": model_df[HONEST_FEATURES].max(),
})
audit

相关性
    HONEST_FEATURES + [TARGET]：列表相加得到 11 个列名（别写成 + TARGET）。
    corr(method="pearson") 算相关矩阵，[TARGET] 只取目标那列。
    drop(TARGET) 去掉自相关 1.0。
    sort_values(...) 排序。相关不是因果。

In [ ]:
corr_pearson = model_df[HONEST_FEATURES + [TARGET]].corr(method="pearson")[TARGET].drop(TARGET)
corr_spearman = model_df[HONEST_FEATURES + [TARGET]].corr(method="spearman")[TARGET].drop(TARGET)
corr_table = pd.DataFrame
({
    "pearson": corr_pearson, "spearman": corr_spearman,
}).sort_values("pearson", ascending=False)
corr_table

标签
    阈值只用训练集算-->防止数据泄露
    (y >= 阈值) 得 True/False，.astype(int) 转 1/0。
    value_counts() 数类别

In [ ]:
train_median = y_train.median()
print("training-median threshold:", train_median)
y_cls_train = (y_train >= train_median).astype(int)
y_cls_test = (y_test >= train_median).astype(int)
print("train class counts:\n", y_cls_train.value_counts())
print("test class counts:\n", y_cls_test.value_counts())

Dummy基线
    DummyRegressor(strategy="mean")`：永远预测训练集均值。回归基线的参照。
    DummyClassifier(strategy="prior")`：永远按训练集的类别比例预测。分类基线的参照
    用和真实模型相同的指标、相同的测试行评估 dummy
    np.sqrt(...) 把 MSE 开方成 RMSE

In [ ]:
dummy_reg = DummyRegressor(strategy="mean").fit(X_train, y_train)
reg_pred_dummy = dummy_reg.predict(X_test)
dummy_cls = DummyClassifier(strategy="prior").fit(X_train, y_cls_train)
cls_pred_dummy = dummy_cls.predict(X_test)
print("dummy reg R2  :", r2_score(y_test, reg_pred_dummy))
print("dummy reg RMSE:", np.sqrt(mean_squared_error(y_test, reg_pred_dummy)))
print("dummy cls F1  :", f1_score(y_cls_test, cls_pred_dummy))

回归CV
    kf.split(X_train) 生成 5 组 (train_idx, val_idx) 位置索引
    iloc[train_idx] 按位置取行。
    每折 reg_pipe.fit 只看该折训练部分 → scaler 看不到验证数据。
    append(...) 收分数；np.round/mean/std 出汇总数据

In [ ]:
reg_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression()),
])
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_r2 = []
fold_rmse = []
for train_idx, val_idx in kf.split(X_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    reg_pipe.fit(X_tr, y_tr)
    pred = reg_pipe.predict(X_val)
    fold_r2.append(r2_score(y_val, pred))
    fold_rmse.append(np.sqrt(mean_squared_error(y_val, pred)))
print("R2 folds  :", np.round(fold_r2, 3))
print("R2 mean/std:", round(np.mean(fold_r2), 3), "/", round(np.std(fold_r2), 3))
print("RMSE folds:", np.round(fold_rmse, 3))
print("RMSE mean/std:", round(np.mean(fold_rmse), 3), "/", round(np.std(fold_rmse), 3))

回归指标
    先用全部训练数据拟合最终模型。
    def 定义函数，return 返回字典；函数体要缩进。
    T 转置，变成行=训练/测试、列=指标

In [ ]:
reg_pipe.fit(X_train, y_train)
def reg_metrics(y_true, y_pred):
    return {
        "R2": r2_score(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "MSE": mean_squared_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
    }
train_metrics = reg_metrics(y_train, reg_pipe.predict(X_train))
test_metrics = reg_metrics(y_test, reg_pipe.predict(X_test))
reg_table = pd.DataFrame({"train": train_metrics, "test": test_metrics}).T
reg_table

回归图
    residual = 真实 - 预测，pandas 按索引对齐后相减。
    scatter 散点图，alpha 半透明。
    plot([lo,hi],[lo,hi],"r--") 画红色虚线 y=x。
    axhline(0, ...) 画 y=0 水平线。
    图像结果：残差围绕 0 随机=健康；弯曲/漏斗/成团=有化学结构没学到。

In [ ]:
test_pred = reg_pipe.predict(X_test)
residual = y_test - test_pred
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(y_test, test_pred, alpha=0.6)
lo, hi = min(y_test.min(), test_pred.min()), max(y_test.max(), test_pred.max())
axes[0].plot([lo, hi], [lo, hi], "r--")
axes[0].set_xlabel("actual gap (eV)")
axes[0].set_ylabel("predicted gap (eV)")
axes[0].set_title("Actual vs predicted (test)")
axes[1].scatter(test_pred, residual, alpha=0.6)
axes[1].axhline(0, color="r", linestyle="--")
axes[1].set_xlabel("predicted gap (eV)")
axes[1].set_ylabel("residual (actual - pred, eV)")
axes[1].set_title("Residuals vs predicted (test)")
plt.tight_layout()
plt.show()

最大误差
    .loc[X_test.index, "id"] 按标签取，把 id 对齐到测试行。
    .abs() 绝对误差
    sort_values(..., ascending=False) 从大到小排序；
    .head(5) 取前 5。

In [ ]:
err_df = pd.DataFrame({
    "id": model_df.loc[X_test.index, "id"],
    "actual": y_test,
    "predicted": test_pred,
    "residual": residual,
    "abs_error": residual.abs(),
}).sort_values("abs_error", ascending=False)
err_df.head(5)

泄漏实验
    对于公式泄漏/预处理泄漏/部署时泄漏，采用先划分后预处理、变换放进pipeline和测试集只在流程定稿最后用一次的方法来防止
    按公式重算能隙，np.abs(...).max() 应接近 0，验证等式成立。
    列表相加得到含泄漏列的特征集；同一划分再拟合一版。
    该段代码输出结果False证明数据没有发生泄漏

In [ ]:
gap_check = model_df["LUMO_eV"] - model_df["HOMO_eV"]
print("max abs difference (LUMO - HOMO - target):",
      np.abs(gap_check - model_df[TARGET]).max())
leaky_features = HONEST_FEATURES + ["HOMO_eV", "LUMO_eV"]
leaky_pipe = Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())])
leaky_pipe.fit(X_train[leaky_features], y_train)
leaky_pred = leaky_pipe.predict(X_test[leaky_features])
print("leaky test R2  :", r2_score(y_test, leaky_pred))
print("leaky test RMSE:", np.sqrt(mean_squared_error(y_test, leaky_pred)))

分类CV
    max_iter=1000 防止数据不收敛。
    StratifiedKFold 保持每折类别比例；.split 要同时传 X 和 y。
    predict 输出 0/1 给 F1；predict_proba 输出 [P(0), P(1)]。


In [ ]:
log_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000)),
])
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_f1 = []
fold_auc = []
for train_idx, val_idx in skf.split(X_train, y_cls_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_cls_train.iloc[train_idx], y_cls_train.iloc[val_idx]
    log_pipe.fit(X_tr, y_tr)
    pred = log_pipe.predict(X_val)
    proba = log_pipe.predict_proba(X_val)[:, 1]
    fold_f1.append(f1_score(y_val, pred))
    fold_auc.append(roc_auc_score(y_val, proba))
print("F1 folds  :", np.round(fold_f1, 3))
print("F1 mean/std:", round(np.mean(fold_f1), 3), "/", round(np.std(fold_f1), 3))
print("AUC folds :", np.round(fold_auc, 3))
print("AUC mean/std:", round(np.mean(fold_auc), 3), "/", round(np.std(fold_auc), 3))

分类指标
    accuracy：总共判对的比例 
    precision 真高能隙的占比
    recall 敏感度
    ROC-AUC 随机挑一个正类、一个负类，模型把正类排前面的概率
    named_steps["model"] 按 Pipeline 里起的名字取出逻辑回归，.n_iter_ 是迭代次数，用于判断收敛。
    训练用 y_cls_train、测试用 y_cls_test，别混。
    index=[...] 手动指定行标签。

In [ ]:
log_pipe.fit(X_train, y_cls_train)
print("converged:", log_pipe.named_steps["model"].n_iter_)
test_pred_cls = log_pipe.predict(X_test)
test_proba_cls = log_pipe.predict_proba(X_test)[:, 1]
cls_table = pd.DataFrame({
    "train": [
        accuracy_score(y_cls_train, log_pipe.predict(X_train)),
        precision_score(y_cls_train, log_pipe.predict(X_train)),
        recall_score(y_cls_train, log_pipe.predict(X_train)),
        f1_score(y_cls_train, log_pipe.predict(X_train)),
        roc_auc_score(y_cls_train, log_pipe.predict_proba(X_train)[:, 1]),
    ],
    "test": [
        accuracy_score(y_cls_test, test_pred_cls),
        precision_score(y_cls_test, test_pred_cls),
        recall_score(y_cls_test, test_pred_cls),
        f1_score(y_cls_test, test_pred_cls),
        roc_auc_score(y_cls_test, test_proba_cls),
    ],
}, index=["accuracy", "precision", "recall", "F1", "ROC-AUC"])
cls_table

混淆矩阵
    返回 [[TN, FP], [FN, TP]]。
    annot=True 写数字，fmt="d" 整数，cmap="Blues" 配色。
    标题写明正类是 high_gap=1，否则分不清 FP/FN。

In [ ]:
cm = confusion_matrix(y_cls_test, test_pred_cls)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["pred 0", "pred 1"],
            yticklabels=["actual 0", "actual 1"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion matrix (test), positive class = high_gap=1")
plt.show()

FP/FN
    FP=真实0预测1；FN=真实1预测0

In [ ]:
inspect = pd.DataFrame({
    "id": model_df.loc[X_test.index, "id"],
    "true_gap": y_test,
    "true_class": y_cls_test,
    "pred_class": test_pred_cls,
    "pred_prob": test_proba_cls,
})
false_positives = inspect[(inspect.true_class == 0) & (inspect.pred_class == 1)]
false_negatives = inspect[(inspect.true_class == 1) & (inspect.pred_class == 0)]
print("false positives:")
print(false_positives.head(2))
print("false negatives:")
print(false_negatives.head(2))

Summary
    把前序计算得到的数据按照作业要求汇集成一张表格
    reg_metrics()["R2"]从字典中提取数据值

In [ ]:
summary = pd.DataFrame({
    "value": [
        len(model_df), len(HONEST_FEATURES), train_median,
        np.mean(fold_r2), np.std(fold_r2),
        reg_metrics(y_test, test_pred)["R2"],
        reg_metrics(y_test, test_pred)["RMSE"],
        np.mean(fold_f1), np.std(fold_f1),
        np.mean(fold_auc), np.std(fold_auc),
        f1_score(y_cls_test, test_pred_cls),
        roc_auc_score(y_cls_test, test_proba_cls),
    ],
}, index=[
    "valid rows", "honest features", "threshold",
    "CV R2 mean", "CV R2 std", "test R2", "test RMSE",
    "CV F1 mean", "CV F1 std", "CV AUC mean", "CV AUC std",
    "test F1", "test AUC",
])
summary

**Question**
1. Are we solving a meaningful prediction problem?
    
    This is a meaningful baseline problem, but it is not a complete materials-discovery model. The target, HOMO_LUMO_Gap_eV, is measured in eV. Each row represents one calculated polymer record. The original dataset contains 1,000 rows. After excluding the 74 rows where Energy_Hatree == 0 as apparent failed calculations, 926 valid rows remain, with 10 honest features.
    
    The inputs are calculated properties such as Energy_Hatree, polarizability, dipole moment, and thermodynamic corrections. The model is therefore a calculation-to-calculation model, not a structure-to-property model. It does not directly read SMILES strings or new polymer structures, and it cannot demonstrate that the model will work for new or not-yet-calculated chemical families. It is appropriate for testing the workflow, leakage, baselines, and error analysis, but it does not directly support a claim that the model can be used to screen new materials.
    
    The classification threshold is the training-set median of the target, 8.33225 eV. It exists only to create a practice binary-classification task; it is not a scientifically validated boundary for the HOMO-LUMO gap. Therefore, high_gap only means “greater than or equal to the median of this dataset’s training rows.” It cannot be interpreted as meaning that a material necessarily has a particular device performance.

2. Does the evaluation measure generalization rather than leakage or memory?
    
    The evaluation provides partial evidence of generalization, but the conclusion must remain limited. The final test set contains 186 rows and is not used during training. 
    The training set contains 740 rows, and the intersection of the training and test indices is 0. StandardScaler is inside the Pipeline, so in each cross-validation fold the scaler is fitted only on that fold’s training portion. The validation rows do not influence the mean or standard deviation used for scaling.
    
    The honest regression model produced a test R² = 0.2494 and RMSE = 0.4519 eV. The dummy regression produced R² = -0.00007 and RMSE = 0.5216 eV. The honest classification model produced a test F1 = 0.7524 and ROC-AUC = 0.7809, while the dummy classifier produced F1 = 0 and ROC-AUC = 0.5. Both honest models beat their dummy baselines, so they learned at least some information beyond the mean or the class prior.
    
    The leakage experiment demonstrates a counterexample. The dataset satisfies:
        HOMO_LUMO_Gap_eV = LUMO_eV - HOMO_eV
   
    Across all 926 valid rows, the maximum absolute deviation is only 1.78e-15, which is floating-point error. When HOMO_eV and LUMO_eV are added back as features, the leaky model achieves R² = 1.0, MAE = 0, and RMSE = 0, even though the test rows were not used for training. This is not predictive ability. The model reconstructs the target directly from the inputs by subtraction. The leaky result must therefore be rejected.
   
    Cross-validation also shows that stability is limited. The five regression R² values are:
        [0.236, 0.143, 0.275, -0.067, 0.235]
    
    One fold performs worse than the training-mean baseline. The five regression RMSE values are:
        [0.429, 0.387, 0.362, 0.539, 0.412]
    
    Regression training R² = 0.2105, while test R² = 0.2494. This does not indicate obvious overfitting, but both scores have limited explanatory power and suggest underfitting. The five classification F1 values range from 0.692 to 0.780, and the five ROC-AUC values range from 0.753 to 0.838. The performance is moderately stable but still varies noticeably.
    
    A random row split estimates generalization only to held-out records from the same data-generating population. It does not estimate performance on genuinely new chemical families, different computational settings, or future out-of-distribution data. These scores are evidence, not proof of performance in future chemical space.


3. What do the scores say—and what do they hide?
    
    The regression test MAE = 0.3011 eV, while RMSE = 0.4519 eV. Both are measured in eV, but RMSE gives more weight to large errors. The fact that RMSE is larger means that a small number of samples have substantial errors. The largest error is for id = 950: the actual value is 4.4689 eV, the prediction is 8.1524 eV, and the residual is approximately -3.6835 eV. This sample pulls the prediction toward the main body of the dataset and substantially increases RMSE. Looking only at the average error therefore understates the main problem.
    
    R² = 0.2494 means that the test-set squared error is about 25% lower than the baseline that always predicts the test mean. It does not mean “25% accuracy.” A negative R² means that the model is worse than the mean baseline. For example, the fourth regression cross-validation fold has R² = -0.0672, meaning that its performance is below that fold’s training-mean baseline.
    